In [2]:
from os.path import realpath

# Local steps to run in pycharm
# conda activate sage
# jupyter lab --no-browser --port=8888
# configure external server with the token
# sagemath (10.7)

from sage.all import *
from sage.plot.plot import graphics_array
from itertools import combinations
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.facecolor': 'white',
})

def graph_square(H):
    G = Graph()
    G.add_vertices(H.vertices())
    for u in H.vertices():
        for v in H.vertices():
            if u < v and H.distance(u,v) <= 2:
                G.add_edge(u,v)
    return G

def is_square_root(G, H):
    return graph_square(H) == G

def find_square_root(G):
    V = list(G.vertices())
    n = len(V)
    pairs = list(combinations(range(n), 2))

    total = 1 << len(pairs)
    for mask in range(total):
        H = Graph()
        H.add_vertices(V)
        for bit, (i, j) in enumerate(pairs):
            if (mask >> bit) & 1:
                H.add_edge(V[i], V[j])
        if is_square_root(G, H):
            return H
    return None

In [20]:
# Efficient find_square_root function

def violates_partial(G, H):
    for u, v in combinations(H.vertices(), 2):
        try:
            d = H.distance(u, v)
        except (ValueError, RuntimeError):
            continue

        if H.has_edge(u, v) and not G.has_edge(u, v):
            return True
        if d <= 2 and not G.has_edge(u, v):
            return True
    return False

def find_square_root(G):
    V = list(G.vertices())
    pairs = list(combinations(V, 2))
    H = Graph()
    H.add_vertices(V)

    def backtrack(idx):
        if idx == len(pairs):
            return H.copy() if is_square_root(G, H) else None

        u, v = pairs[idx]

        H.add_edge(u, v)
        if not violates_partial(G, H):
            result = backtrack(idx + 1)
            if result:
                return result
        H.delete_edge(u, v)

        if not violates_partial(G, H):
            result = backtrack(idx + 1)
            if result:
                return result

        return None

    return backtrack(0)

In [21]:
# LP with min objective function
def min_linear_program(G):
    m = MixedIntegerLinearProgram(solver = 'GLPK', maximization=False)
    m.solver_parameter("simplex_or_intopt", "intopt_only")

    x = m.new_variable(real=True, nonnegative=True)
    y = m.new_variable(real=True, nonnegative=True)
    z = m.new_variable(real=True, nonnegative=True)

    A_G = G.adjacency_matrix()
    n = A_G.nrows()
    M = n**2

    # Indicator argument
    for i in range(n):
        for j in range(n):
            if i == j:
                m.add_constraint(z[i,j] - (sum(y[i,k,j] for k in range(n)) - sum(x[i,k] for k in range(n)) + x[i,j]) == 0)
            else:
                m.add_constraint(z[i,j] - (sum(y[i,k,j] for k in range(n)) + x[i,j]) == 0)

    # Indicator function
    for i in range(n):
        for j in range(n):
            m.add_constraint(z[i,j] - M * A_G[i][j] <= 0)
            m.add_constraint(-z[i,j] + A_G[i][j] <= 0)

    # Product
    for i in range(n):
        for j in range(n):
            for k in range(n):
                m.add_constraint(y[i,k,j] - x[i,k] <= 0)
                m.add_constraint(y[i,k,j] - x[k,j] <= 0)
                m.add_constraint(- y[i,k,j] + x[i,k] + x[k,j] - 1 <= 0)

    # Symmetry
    for i in range(n):
        for j in range(n):
            m.add_constraint(x[i,j] == x[j,i])

    # Diagonal
    for i in range(n):
        m.add_constraint(x[i,i] == 0)

    # Variables
    for i in range(n):
        for j in range(n):
            m.add_constraint(x[i,j] - 1 <= 0)
            for k in range(n):
                m.add_constraint(y[i,k,j] - 1 <= 0)

    m.set_objective(- m.sum(x[i,j] for i in range(n) for j in range(n)))
    return m, x

# LP with min objective function and additional triangle constraints for line graph root structure
def min_triangular_linear_program(G):
    m = MixedIntegerLinearProgram(solver = 'GLPK', maximization=False)
    m.solver_parameter("simplex_or_intopt", "intopt_only")

    x = m.new_variable(real=True, nonnegative=True)
    y = m.new_variable(real=True, nonnegative=True)
    z = m.new_variable(real=True, nonnegative=True)

    A_G = G.adjacency_matrix()
    n = A_G.nrows()
    M = n**2

        # Indicator argument
    for i in range(n):
        for j in range(n):
            if i == j:
                m.add_constraint(z[i,j] - (sum(y[i,k,j] for k in range(n)) - sum(x[i,k] for k in range(n)) + x[i,j]) == 0)
            else:
                m.add_constraint(z[i,j] - (sum(y[i,k,j] for k in range(n)) + x[i,j]) == 0)

    # Indicator function
    for i in range(n):
        for j in range(n):
            m.add_constraint(z[i,j] - M * A_G[i][j] <= 0)
            m.add_constraint(-z[i,j] + A_G[i][j] <= 0)

    # Product
    for i in range(n):
        for j in range(n):
            for k in range(n):
                m.add_constraint(y[i,k,j] - x[i,k] <= 0)
                m.add_constraint(y[i,k,j] - x[k,j] <= 0)
                m.add_constraint(- y[i,k,j] + x[i,k] + x[k,j] - 1 <= 0)

    # Symmetry
    for i in range(n):
        for j in range(n):
            m.add_constraint(x[i,j] == x[j,i])

    # Diagonal
    for i in range(n):
        m.add_constraint(x[i,i] == 0)

    # Variables
    for i in range(n):
        for j in range(n):
            m.add_constraint(x[i,j] - 1 <= 0)
            for k in range(n):
                m.add_constraint(y[i,k,j] - 1 <= 0)

    # Count triangles and consider vertices in only one triangle O(n^3)
    deg = {v: G.degree(v) for v in range(n)}
    triangles_at = {v: [] for v in range(n)}

    for i in range(n):
        for j in range(i+1, n):
            if not G.has_edge(i, j):
                continue
            for k in range(j+1, n):
                if G.has_edge(i, k) and G.has_edge(j, k):
                    triangles_at[i].append((j, k))
                    triangles_at[j].append((i, k))
                    triangles_at[k].append((i, j))

    for s, tris in triangles_at.items():
        if len(tris) == 1:
            u, v = tris[0]
            if deg[u] < deg[v]:
                w = u
            elif deg[v] < deg[u]:
                w = v
            else:
                continue
            m.add_constraint(x[s,w] == 1)
            m.add_constraint(x[w,s] == 1)

    m.set_objective(- m.sum(x[i,j] for i in range(n) for j in range(n)))
    return m, x

In [22]:
# Generate all possible graph squares/roots and their complements for given n
import os
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from sage.graphs.graph_generators import graphs as graph_gens
from sage.numerical.mip import MIPSolverException

n = 7
ncpus = 12
dir = f"min_triangular_complement_graph_output"
output_dir = os.path.join(dir, f"n={n}")
output_pdf = os.path.join(output_dir, f"graph_results_n={n}.pdf")
comparison_pdf = os.path.join(output_dir, f"graph_comparison_n={n}.pdf")
summary_file = os.path.join(output_dir, f"graph_summary_n={n}.txt")

os.makedirs(output_dir, exist_ok=True)
os.makedirs(dir, exist_ok=True)

def error_plot(message):
    g = Graphics()
    g += text(message, (0, 0), fontsize=14, color="red", horizontal_alignment='center')
    g.axes(False)
    return g

def fmt(x, width=8, digits=3):
    if x is None:
        return f"{'NA':>{width}}"
    return f"{x:>{width}.{digits}f}"

def process_graph(idx, G, label="Original"):
    G.relabel(range(G.order()))
    n = G.order()

    A_H = None
    sol_H = None
    H = None
    lp_status = "unknown"
    has_root = False
    edges_H = None
    is_noninteger = False

    try:
        m, x = min_triangular_linear_program(G)
        m.solve()
        solution_x = m.get_values(x)
        A_H = matrix([[solution_x[i, j] for j in range(n)] for i in range(n)])
        sol_H = Graph(A_H, weighted=True)
        for (u, v, w) in sol_H.edges():
            sol_H.set_edge_label(u, v, round(w, 3))

        p1 = sol_H.plot(edge_labels=True, title='Weighted LP H')
        lp_status = "feasible"

        if any(abs(A_H[i, j] - round(A_H[i, j])) > 1e-6 for i in range(n) for j in range(n)):
            is_noninteger = True

    except MIPSolverException:
        sol_H = None
        p1 = error_plot("LP infeasible")
        lp_status = "infeasible"

    H = find_square_root(G)

    if H is None:
        p2 = error_plot("No feasible H")
        has_root = False
        edges_H = None
    else:
        p2 = H.plot(title='Feasible H')
        has_root = True
        edges_H = H.size()

    p3 = G.plot(title=f"{label} G (#{idx})")

    summary = {
        "index": idx,
        "type": label,
        "edges_G": G.size(),
        "vertices_G": n,
        "lp_status": lp_status,
        "has_root": has_root,
        "edges_H": edges_H,
        "is_noninteger": is_noninteger
    }

    is_mismatch = (is_noninteger or (lp_status == "feasible" and H is None))

    if is_mismatch:
        if sol_H is not None:
            p_left = sol_H.plot(edge_labels=True, title="Weighted LP H")
        else:
            p_left = error_plot("No LP solution")

        if H is not None:
            p_mid = H.plot(title="Feasible H")
        else:
            p_mid = error_plot("No discrete H")

        p_right = G.plot(title="Original G")
        comparison = graphics_array([p_left, p_mid, p_right])
    else:
        comparison = None

    return (p1, p2, p3), summary, comparison, is_mismatch

def run_task(idx, G):
    return process_graph(idx, G)

# seen = set()
# tasks = []
# for i, G0 in enumerate(graph_gens.nauty_geng(str(n))):
#     G = Graph(G0)
#     G.relabel(range(G.order()))
#
#     canon = G.canonical_label().graph6_string()
#
#     if canon in seen:
#         continue
#
#     tasks.append((i, G))
#     seen.add(canon)
#
#     Gc = G.complement()
#     seen.add(Gc.canonical_label().graph6_string())

tasks = [(i, Graph(G0)) for i, G0 in enumerate(graph_gens.nauty_geng(str(n)))]

results = []
mismatch_results = []

for idx, G in tasks:

    plot_G, summary_G, comp_G, mismatch_G = process_graph(idx, G, label="Original")
    if mismatch_G and comp_G is not None:
        mismatch_results.append((comp_G, summary_G))

    Gc = G.complement()
    Gc.relabel(range(Gc.order()))

    plot_Gc, summary_Gc, comp_Gc, mismatch_Gc = process_graph(idx, Gc, label="Complement")
    if mismatch_Gc and comp_Gc is not None:
        mismatch_results.append((comp_Gc, summary_Gc))

    (p1, p2, p3) = plot_G
    (q1, q2, q3) = plot_Gc

    combined = graphics_array([
        [p1, p2, p3],
        [q1, q2, q3]
    ])

    results.append((combined, summary_G, summary_Gc))


    print(
        f"✅ Graph {idx:>4} | "
        f"Original LP={summary_G['lp_status']} Root={summary_G['has_root']} | "
        f"Complement LP={summary_Gc['lp_status']} Root={summary_Gc['has_root']}"
    )


with open(summary_file, "w") as f:

    header = (
        f"{'Graph':>5} | {'Type':>10} | {'Verts':>5} | "
        f"{'Edges(G)':>8} | {'Edges(H)':>8} | "
        f"{'LP':>10} | {'Root':>5} | {'NonInt':>7}\n"
    )

    f.write("SUMMARY\n")
    f.write(header)
    f.write("-" * len(header) + "\n")

    for _, sG, sGc in results:
        f.write(
            f"{sG['index']:>5} | "
            f"{sG['type']:>10} | "
            f"{sG['vertices_G']:>5} | "
            f"{sG['edges_G']:>8} | "
            f"{str(sG['edges_H']) if sG['edges_H'] is not None else 'NA':>8} | "
            f"{sG['lp_status']:<10} | "
            f"{'Yes' if sG['has_root'] else 'No':>5} | "
            f"{'Yes' if sG['is_noninteger'] else 'No':>7}\n"
        )

        f.write(
            f"{'':>5} | "
            f"{sGc['type']:>10} | "
            f"{sGc['vertices_G']:>5} | "
            f"{sGc['edges_G']:>8} | "
            f"{str(sGc['edges_H']) if sGc['edges_H'] is not None else 'NA':>8} | "
            f"{sGc['lp_status']:<10} | "
            f"{'Yes' if sGc['has_root'] else 'No':>5} | "
            f"{'Yes' if sGc['is_noninteger'] else 'No':>7}\n"
        )

        f.write("-" * len(header) + "\n")

print(f"📝 Summary written to {summary_file}")

with PdfPages(output_pdf) as pdf:
    for p, sG, sGc in results:
        fig = p.matplotlib(figsize=(12, 4))
        pdf.savefig(fig)
        plt.close(fig)
print(f"\n✅ Combined {len(results)} graphs into {output_pdf}")

if mismatch_results:
    with PdfPages(comparison_pdf) as pdf:
        for comp, s in mismatch_results:

            fig = comp.matplotlib(figsize=(12, 4))

            plt.suptitle(
                f"Graph {s['index']} — "
                f"LP={s['lp_status']} | "
                f"Root={s['has_root']} | "
                f"{'Non-integer LP' if s['is_noninteger'] else ''}"
            )

            pdf.savefig(fig)
            plt.close(fig)

    print(f"⚠️ Comparison mismatches saved to {comparison_pdf}")

✅ Graph    0 | Original LP=feasible Root=True | Complement LP=feasible Root=True
✅ Graph    1 | Original LP=feasible Root=True | Complement LP=feasible Root=True
✅ Graph    2 | Original LP=infeasible Root=False | Complement LP=feasible Root=True
✅ Graph    3 | Original LP=infeasible Root=False | Complement LP=feasible Root=True
✅ Graph    4 | Original LP=infeasible Root=False | Complement LP=feasible Root=True
✅ Graph    5 | Original LP=infeasible Root=False | Complement LP=infeasible Root=False
✅ Graph    6 | Original LP=infeasible Root=False | Complement LP=feasible Root=True
✅ Graph    7 | Original LP=feasible Root=True | Complement LP=feasible Root=True
✅ Graph    8 | Original LP=infeasible Root=False | Complement LP=feasible Root=True
✅ Graph    9 | Original LP=infeasible Root=False | Complement LP=feasible Root=True
✅ Graph   10 | Original LP=feasible Root=True | Complement LP=feasible Root=True
✅ Graph   11 | Original LP=infeasible Root=False | Complement LP=feasible Root=True
✅

<Figure size 640x480 with 0 Axes>

In [2]:
# Analyze graph summary with Original vs Complement separation

filename = "min_complement_graph_output/n=6/graph_summary_n=6.txt"

summary = {}
by_type = {"Original": {}, "Complement": {}}

fractional_with_root = {"Original": [], "Complement": []}
fractional_without_root = {"Original": [], "Complement": []}

count_feasible_root_no = {"Original": 0, "Complement": 0}
violations = []

with open(filename, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if (not line or
            line.startswith('-') or
            line.startswith('Graph') or
            line.startswith('SUMMARY')):
            continue

        parts = [p.strip() for p in line.split('|')]

        if len(parts) != 8:
            continue

        graph, typ, verts, edges_g, edges_h, lp, root, nonint = parts

        H_exists = (root == "Yes")

        if lp == "infeasible":
            LP_type = "LP infeasible"
        elif nonint == "Yes":
            LP_type = "LP fractional"
        else:
            LP_type = "LP integer"

        key = (typ, H_exists, LP_type)
        summary.setdefault(key, []).append(graph)

        by_type.setdefault(typ, {})
        by_type[typ].setdefault((H_exists, LP_type), 0)
        by_type[typ][(H_exists, LP_type)] += 1

        if lp == "feasible" and root == "No":
            count_feasible_root_no[typ] += 1

        if lp == "infeasible" and root != "No":
            violations.append((graph, typ))

        if LP_type == "LP fractional":
            if H_exists:
                fractional_with_root[typ].append(graph)
            else:
                fractional_without_root[typ].append(graph)

for (typ, H_exists, LP_type), graphs in sorted(summary.items()):
    print(
        f"{typ:<11} | "
        f"H exists: {str(H_exists):<5} | "
        f"{LP_type:<15} → Count: {len(graphs)}"
    )

for typ in ["Original", "Complement"]:
    print(f"\n--- {typ} ---")
    for (H_exists, LP_type), count in sorted(by_type.get(typ, {}).items()):
        print(
            f"H exists: {str(H_exists):<5} | "
            f"{LP_type:<15} → {count}"
        )

Complement  | H exists: False | LP fractional   → Count: 21
Complement  | H exists: False | LP infeasible   → Count: 107
Complement  | H exists: True  | LP fractional   → Count: 6
Complement  | H exists: True  | LP integer      → Count: 22
Original    | H exists: False | LP fractional   → Count: 21
Original    | H exists: False | LP infeasible   → Count: 107
Original    | H exists: True  | LP fractional   → Count: 4
Original    | H exists: True  | LP integer      → Count: 24

--- Original ---
H exists: False | LP fractional   → 21
H exists: False | LP infeasible   → 107
H exists: True  | LP fractional   → 4
H exists: True  | LP integer      → 24

--- Complement ---
H exists: False | LP fractional   → 21
H exists: False | LP infeasible   → 107
H exists: True  | LP fractional   → 6
H exists: True  | LP integer      → 22
